# 16 — RAG, ACL nativa e inferência no Runtime

Referência: [AgentCore workshops](https://github.com/awslabs/agentcore-samples/tree/main/06-workshops). O código e o IaC desta jornada adaptam os conceitos às contas do workshop.

## Pré-requisito: 1 conta AWS

Esta aula não exige fronteiras entre contas. Seus stacks, integrações e permissões precisam estar implantados na conta da plataforma. Ela não funciona automaticamente em uma conta AWS vazia.

**Runner CodeBuild do projeto:** a seleção é declarada no deploy. A jornada `single-account` inclui 00–09 e 15–16; Registry, Terraform e RAG são exigidos somente pelas aulas que os utilizam. Ampliar a seleção exige atualizar a implantação pelo IaC. Confira a [matriz de contas e execução](../docs/contas-e-elegibilidade.md).

![Arquitetura AWS da aula 16](../docs/diagramas/svg/16_rag_managed_identity_policy.svg)

Leia as fronteiras do diagrama e confronte-as com o IaC e as chamadas abaixo.

Todos os serviços e agentes desta jornada são implantados na AWS.
O notebook é o roteiro e cliente das APIs; ferramentas, inferência,
políticas, fontes e memória executam nos serviços indicados.
Use o ambiente AWS descrito no [guia de implantação](../docs/deploy.md).
Configure os identificadores dos stacks antes de executar. Uma
dependência ausente bloqueia a aula antes das chamadas do exercício.
Antes de usar “Run All”, selecione as aulas compatíveis com as contas
preparadas. O bloco 10–14 exige quatro contas na implantação atual.

### Carregar a configuração e conferir as dependências

A configuração contém perfis/roles e IDs, nunca senhas ou tokens. O STS confirma a conta antes de consultar os stacks; o manifesto exige as dependências desta aula.

In [ ]:
import json
import os
import uuid
from pathlib import Path
from IPython.display import Code, display
import inspect
from santander_demo.aws_workshop.client import AWSWorkshop
from santander_demo.aws_workshop.config import load
from santander_demo.aws_workshop.manifest import require

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
CONFIG = Path(os.environ.get("WORKSHOP_CONFIG", ROOT / "config/aws.workshop.json"))
config = load(CONFIG)
dependencies = require(config, "16")
print({"lab": "16", "dependencies": list(dependencies), "region": config["region"]})

### Ler o conector nativo e a ingestão com ACL

A base MANAGED usa documentos S3 com uma ACL global. O Gateway expõe o conector nativo e Cedar vincula userContext.userId ao email do JWT. O modelo só recebe trechos autorizados.

In [ ]:
from santander_demo.rag.deployment import outputs
from santander_demo.rag.cognito import login
from santander_demo.aws_workshop.config import session, stack_outputs
from santander_demo.aws_workshop.client import stack_name
rag_config = json.loads((ROOT / dependencies["rag"]["config"]).read_text())
aws = session(config)
identity = outputs(aws, rag_config, "identity")
data = outputs(aws, rag_config, "data")
access = outputs(aws, rag_config, "access")
runtime = stack_outputs(aws, stack_name(config, "platform", "rag-agent"))
display(Code((ROOT / "infra/rag/templates/data.json").read_text(), language="json"))
kb = aws.client("bedrock-agent").get_knowledge_base(knowledgeBaseId=data["KnowledgeBaseId"])
assert kb["knowledgeBase"]["status"] == "ACTIVE"
print({"knowledge_base": data["KnowledgeBaseId"], "runtime": runtime["RuntimeArn"]})

### Ler o código da recuperação e da geração no Runtime

O Runtime deriva a identidade do token, obtém um WAT, chama o Gateway e só então invoca Bedrock Converse. A role do Runtime tem uma negativa explícita de retrieval direto.

In [ ]:
from santander_demo.aws_workshop.rag_agent import run
from santander_demo.rag.client import ask
display(Code(inspect.getsource(run), language="python"))
display(Code(inspect.getsource(ask), language="python"))

### Comprovar a negativa de retrieval direto

Este probe executa com a role do próprio Runtime. Uma tentativa de Retrieve sem passar pelo Gateway deve receber AccessDeniedException do IAM. Timeout ou outro erro não contam como negativa.

In [ ]:
import httpx
from urllib.parse import quote
url = (f"https://bedrock-agentcore.{config['region']}.amazonaws.com/runtimes/"
       + quote(runtime["RuntimeArn"], safe="") + "/invocations?qualifier=DEFAULT")
async with httpx.AsyncClient(timeout=240, follow_redirects=False) as http:
    response = await http.post(url, headers={
        "Authorization": "Bearer " + login(aws, rag_config, identity, "c"),
        "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": str(uuid.uuid4()),
    }, json={"operation": "probe_direct_retrieval"})
    response.raise_for_status()
    probe = response.json()
assert probe == {"execution": "AWS_AGENTCORE_RUNTIME", "direct_retrieval": "IAM_DENY"}
print(probe)

### Tentar fornecer a identidade de outra pessoa

Enviamos um JWT válido de A com userContext de C. A Policy deve negar a inconsistência antes da consulta; esta chamada sai do runner AWS e mantém os IDs nativos de requisição do Gateway.

In [ ]:
from santander_demo.rag.client import probe_identity_spoof
import time
since_ms = int(time.time() * 1000)
spoof = await probe_identity_spoof(
    rag_config, access, login(aws, rag_config, identity, "a"),
    rag_config["users"]["c"]["email"],
)
assert spoof["outcome"] == "POLICY_DENY_REPORTED"
assert spoof["gateway_request_ids"]
print(spoof)

### Fazer a mesma pergunta como A, B, C e D

A e B recebem apenas documentos de suas áreas; C pode receber ambas; D deve ser negado antes da geração. Todas as chamadas de modelo e retrieval acontecem no Runtime.

In [ ]:
results = {}
allowed = {"a": {"R1", "R2"}, "b": {"F1", "F2"}, "c": {"R1", "R2", "F1", "F2"}, "d": set()}
async with httpx.AsyncClient(timeout=240, follow_redirects=False) as http:
    for label in "abcd":
        jwt_token = login(aws, rag_config, identity, label)
        response = await http.post(url, headers={"Authorization": "Bearer " + jwt_token,
            "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": str(uuid.uuid4())},
            json={"question": "Quais são as políticas de riscos e fraude disponíveis?"})
        response.raise_for_status()
        result = response.json()
        assert result["execution"] == "AWS_AGENTCORE_RUNTIME"
        assert result["gateway_request_ids"]
        assert set(result["document_ids"]) <= allowed[label]
        if label == "d":
            assert result["outcome"] == "POLICY_DENY_REPORTED" and not result["generated"]
        else:
            assert result["document_ids"] and result["generated"]
            assert result["model_request_id"] and result["citations"]
            assert set(result["citations"]) <= set(result["document_ids"])
            assert {chunk["id"] for chunk in result["retrieved_context"]} == set(result["document_ids"])
        results[label] = result
print(json.dumps(results, indent=2, ensure_ascii=False))

### Verificar cobertura do corpus e decisões nativas

A busca pelos quatro títulos precisa alcançar todos os documentos permitidos para cada pessoa. O probe executa no mesmo Runtime, sem geração adicional. Correlacionamos cada Retrieve com o Gateway e sua decisão Policy antes de aceitar a matriz.

In [ ]:
import asyncio
from santander_demo.rag.client import acceptance
from santander_demo.rag.transport import correlate
async with httpx.AsyncClient(timeout=240, follow_redirects=False) as http:
    for label in "abc":
        response = await http.post(url, headers={
            "Authorization": "Bearer " + login(aws, rag_config, identity, label),
            "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": str(uuid.uuid4()),
        }, json={"operation": "coverage"})
        response.raise_for_status()
        coverage = response.json()
        assert coverage["execution"] == "AWS_AGENTCORE_RUNTIME"
        assert set(coverage["coverage_document_ids"]) == allowed[label]
        results[label].update(coverage)
rows = [*results.values(), spoof]
for attempt in range(48):
    correlate(aws, access, rows, since_ms)
    if all(row["gateway_log_correlation_verified"] for row in rows):
        break
    await asyncio.sleep(5)
else:
    raise AssertionError("RAG_NATIVE_POLICY_LOGS_MISSING")
report = acceptance(rows)
assert report["status"] == "PASS", report
print(json.dumps({"acceptance": report, "evidence": rows}, indent=2, ensure_ascii=False))

**Entra ID:** identidade, Policy e ACL cumprem os mesmos papéis, mas o provider OAuth, a troca OBO e as claims são próprios do Entra. A fronteira final continua sendo a ACL nativa da fonte.

### Evidências e encerramento

Registre IDs de requisição e decisões, sem tokens ou senhas. Os recursos são compartilhados entre aulas: a remoção pertence ao pipeline de limpeza do projeto e deve respeitar essa dependência.